# Telco Customer Churn Prediction

**Assignment:** End-to-end churn prediction for a telecommunications company.

This notebook covers the required workflow **Business Problem → Data → Preparation → EDA → Feature Engineering → Model → Evaluation → Interpretation → Saved Model → API**, plus bonus work: class-imbalance handling, hyperparameter tuning, and additional model comparison.

## 1. Business Problem

The business wants to identify customers who are likely to churn so the retention team can intervene proactively. Because missing a true churner can be costly, recall is important; however, excessive false positives also waste retention resources. We therefore report precision, recall and F1 and use **F1** as the primary tuning metric.

In [ ]:
import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             average_precision_score, RocCurveDisplay, PrecisionRecallDisplay)
from model_utils import FeatureEngineer
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_PATH = 'data/TelcoCustomerChurn.csv'
df = pd.read_csv(DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Shape:', df.shape)
display(df.head())


## 2. Data Understanding & Preparation
We inspect structure, data types, missing values, duplicates, numerical/categorical columns and target distribution. `TotalCharges` contains blank strings in the raw dataset, so they are converted to missing numeric values and later imputed **inside the training pipeline** to avoid leakage.

In [ ]:
print(df.info())
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Duplicate rows:', df.duplicated().sum())
print('Unique customer IDs:', df['customerID'].nunique())
print('\nTarget distribution:')
display(df['Churn'].value_counts().to_frame('count'))
display((df['Churn'].value_counts(normalize=True)*100).round(2).to_frame('percent'))
num_cols_raw = df.select_dtypes(include=np.number).columns.tolist()
cat_cols_raw = df.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric columns:', num_cols_raw)
print('Categorical columns:', cat_cols_raw)


In [ ]:
plt.figure(figsize=(6,4)); df['Churn'].value_counts().plot(kind='bar'); plt.title('1. Churn Distribution'); plt.xlabel('Churn'); plt.ylabel('Customers'); plt.xticks(rotation=0); plt.show()
print('Business insight: The target is imbalanced, so accuracy alone can be misleading; minority-class recall and F1 should be monitored.')

In [ ]:
plt.figure(figsize=(7,4)); df.boxplot(column='tenure', by='Churn'); plt.title('2. Tenure by Churn'); plt.suptitle(''); plt.xlabel('Churn'); plt.ylabel('Tenure (months)'); plt.show()
print('Business insight: Churners tend to have shorter tenure, suggesting early-lifecycle retention interventions may be valuable.')

In [ ]:
pd.crosstab(df['Contract'],df['Churn']).plot(kind='bar',figsize=(8,4)); plt.title('3. Churn by Contract Type'); plt.xlabel('Contract'); plt.ylabel('Customers'); plt.xticks(rotation=15); plt.show()
print('Business insight: Month-to-month customers show substantially more churn than customers on longer contracts.')

In [ ]:
pd.crosstab(df['InternetService'],df['Churn']).plot(kind='bar',figsize=(8,4)); plt.title('4. Churn by Internet Service'); plt.xlabel('Internet Service'); plt.ylabel('Customers'); plt.xticks(rotation=0); plt.show()
print('Business insight: Churn varies by internet service type, indicating service/product experience is associated with retention risk.')

In [ ]:
plt.figure(figsize=(8,4)); df.loc[df['Churn']=='No','MonthlyCharges'].plot(kind='kde',label='No Churn'); df.loc[df['Churn']=='Yes','MonthlyCharges'].plot(kind='kde',label='Churn'); plt.title('5. Monthly Charges by Churn'); plt.legend(); plt.show()
print('Business insight: Churners are more concentrated in some higher monthly-charge ranges, so pricing/service value may matter.')

In [ ]:
pd.crosstab(df['PaymentMethod'],df['Churn']).plot(kind='bar',figsize=(8,4)); plt.title('6. Churn by Payment Method'); plt.xlabel('Payment Method'); plt.ylabel('Customers'); plt.xticks(rotation=25); plt.show()
print('Business insight: Churn differs across payment methods, which can help target billing/payment-experience improvements.')

## 3. Feature Engineering
The feature engineering transformer is fitted without using the target and is placed **inside the modelling pipeline**. This ensures the same transformations are applied to unseen API customers.

1. **AvgMonthlyCharge** = TotalCharges / max(tenure, 1): an approximate historical monthly charge signal.
2. **NumServices** = count of subscribed services marked `Yes`: a simple service-adoption measure.
3. **TenureGroup** = interpretable tenure bands: 0–6, 7–12, 13–24, 25–48 and 49+ months.

In [ ]:
from model_utils import FeatureEngineer
fe = FeatureEngineer()
df_fe = fe.transform(df.drop(columns=['customerID','Churn']))
display(df_fe[['tenure','TotalCharges','AvgMonthlyCharge','NumServices','TenureGroup']].head())


## 4. Train/Test Split and Leakage Prevention
A **70:30 stratified split** is used with `random_state=42`, exactly as required. The test set is held out until final evaluation. All imputing, encoding, feature engineering and scaling (where applicable) are learned only from training data.

In [ ]:
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['customerID','Churn'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Train churn rate:', round(y_train.mean(),4), 'Test churn rate:', round(y_test.mean(),4))


In [ ]:
numeric_features = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges','AvgMonthlyCharge','NumServices']
categorical_features = ['gender','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod','TenureGroup']
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
def make_pipe(model):
    return Pipeline([('features', FeatureEngineer()), ('prep', preprocessor), ('clf', model)])


## 5. Required Decision Tree Modelling + Class Imbalance
We compare an ordinary Decision Tree with a class-balanced Decision Tree. `class_weight='balanced'` automatically gives greater weight to the minority churn class. This is a meaningful bonus activity because the target distribution is not balanced.

In [ ]:
models = {
    'Decision Tree - baseline': make_pipe(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    'Decision Tree - balanced': make_pipe(DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test); prob = model.predict_proba(X_test)[:,1]
    results[name] = {
        'Accuracy': accuracy_score(y_test,pred), 'Precision': precision_score(y_test,pred,zero_division=0),
        'Recall': recall_score(y_test,pred,zero_division=0), 'F1': f1_score(y_test,pred,zero_division=0),
        'ROC-AUC': roc_auc_score(y_test,prob), 'PR-AUC': average_precision_score(y_test,prob)
    }
comparison = pd.DataFrame(results).T.round(4)
display(comparison)
print('Interpretation: class balancing is assessed by its effect on minority-class recall and F1, not just accuracy.')


## 6. Hyperparameter Tuning (Bonus)
A 5-fold `GridSearchCV` is performed on the training set only. The search optimizes F1 and tunes tree depth and minimum sample constraints. The untouched test set is used only after the best configuration is selected.

In [ ]:
tuned_pipe = make_pipe(DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
param_grid = {
    'clf__max_depth': [3,5,7,10],
    'clf__min_samples_leaf': [5,10],
    'clf__min_samples_split': [2,10]
}
grid = GridSearchCV(tuned_pipe, param_grid, cv=5, scoring='f1', n_jobs=2, return_train_score=False)
grid.fit(X_train, y_train)
print('Best CV F1:', round(grid.best_score_,4))
print('Best parameters:', grid.best_params_)
best_dt = grid.best_estimator_


## 7. Additional Model Comparison (Bonus)
To demonstrate that the final choice is not arbitrary, we compare the tuned Decision Tree with a class-balanced Logistic Regression and Random Forest. The Decision Tree remains the required primary model, while the additional models provide a useful benchmark.

In [ ]:
extra_models = {
    'Logistic Regression - balanced': make_pipe(LogisticRegression(max_iter=1000, solver='liblinear', class_weight='balanced', random_state=RANDOM_STATE)),
    'Random Forest - balanced': make_pipe(RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    'Decision Tree - tuned + balanced': best_dt
}
for name, model in extra_models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
    results[name]={'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob),'PR-AUC':average_precision_score(y_test,prob)}
all_results=pd.DataFrame(results).T.sort_values('F1',ascending=False).round(4)
display(all_results)
print('Final selection principle: retain the required Decision Tree while preferring the tuned configuration because it is interpretable, leakage-safe, class-balanced and selected by cross-validated F1.')


## 8. Final Model Evaluation
The following metrics are reported for the selected tuned Decision Tree: Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC and PR-AUC.

In [ ]:
y_pred = best_dt.predict(X_test); y_prob = best_dt.predict_proba(X_test)[:,1]
print(classification_report(y_test,y_pred,target_names=['No Churn','Churn'],digits=4))
cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(5,4)); plt.imshow(cm, interpolation='nearest'); plt.title('Confusion Matrix - Final Decision Tree'); plt.colorbar(); plt.xticks([0,1],['No Churn','Churn']); plt.yticks([0,1],['No Churn','Churn']); plt.xlabel('Predicted'); plt.ylabel('Actual');
for i in range(2):
    for j in range(2): plt.text(j,i,str(cm[i,j]),ha='center',va='center');
plt.show()
print('Accuracy:',round(accuracy_score(y_test,y_pred),4))
print('Precision:',round(precision_score(y_test,y_pred,zero_division=0),4))
print('Recall:',round(recall_score(y_test,y_pred,zero_division=0),4))
print('F1:',round(f1_score(y_test,y_pred,zero_division=0),4))
print('ROC-AUC:',round(roc_auc_score(y_test,y_prob),4))
print('PR-AUC:',round(average_precision_score(y_test,y_prob),4))


In [ ]:
fig=plt.figure(figsize=(7,5)); RocCurveDisplay.from_predictions(y_test,y_prob); plt.title('ROC Curve - Final Model'); plt.show()

In [ ]:
fig=plt.figure(figsize=(7,5)); PrecisionRecallDisplay.from_predictions(y_test,y_prob); plt.title('Precision-Recall Curve - Final Model'); plt.show()

### Precision vs Recall — Business Decision
For churn prevention, **Recall is generally the priority** because a false negative means failing to identify a customer who actually churns. A retention team can tolerate some false positives if outreach costs are manageable. Precision still matters because very low precision can make campaigns inefficient. The F1 score therefore provides a balanced model-selection criterion.

## 9. Model Interpretation
We inspect the most important features used by the final Decision Tree and visualize a shallow view of the tree for human interpretation.

In [ ]:
feature_names = best_dt.named_steps['prep'].get_feature_names_out()
importances = best_dt.named_steps['clf'].feature_importances_
fi = pd.DataFrame({'Feature':feature_names,'Importance':importances}).sort_values('Importance',ascending=False).head(15)
display(fi)
plt.figure(figsize=(9,6)); plt.barh(fi['Feature'][::-1], fi['Importance'][::-1]); plt.title('Top 15 Feature Importances'); plt.xlabel('Importance'); plt.show()
print('Business insight: The most influential signals show which customer/account characteristics the tree uses most strongly when separating churn risk.')


In [ ]:
plt.figure(figsize=(20,10))
plot_tree(best_dt.named_steps['clf'], feature_names=feature_names, class_names=['No Churn','Churn'], max_depth=3, filled=False, fontsize=7)
plt.title('Final Decision Tree - First 3 Levels')
plt.show()


## 10. Save the Final Pipeline
The saved artifact contains feature engineering, imputers, one-hot encoding and the classifier in one object. This prevents training/inference preprocessing mismatch.

In [ ]:
os.makedirs('model',exist_ok=True)
joblib.dump(best_dt,'model/churn_pipeline.pkl')
fi_full=pd.DataFrame({'feature':feature_names,'importance':importances}).sort_values('importance',ascending=False)
fi_full.to_csv('model/feature_importance.csv',index=False)
with open('model/metrics.json','w') as f:
    json.dump({'best_params':grid.best_params_,'cv_best_f1':float(grid.best_score_),'test_metrics':results['Decision Tree - tuned + balanced']},f,indent=2)
print('Saved: model/churn_pipeline.pkl, model/feature_importance.csv, model/metrics.json')


## 11. API Readiness
The FastAPI application loads the same saved pipeline and accepts customer JSON at `POST /predict`. Invalid fields are rejected by Pydantic and prediction errors return HTTP 400. Because the full feature-engineering/preprocessing pipeline is serialized, the API does not need a separate hand-written preprocessing implementation.

In [ ]:
# Optional local smoke test (requires the API server to be running):
# curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" --data @sample_request.json
print('API files: app.py + sample_request.json')


## 12. Final Conclusions
- The required Decision Tree workflow is complete with a reproducible 70:30 stratified split and `random_state=42`.
- The analysis explicitly addresses class imbalance.
- Hyperparameter tuning uses 5-fold CV on the training data only.
- Logistic Regression and Random Forest provide additional model benchmarks.
- Precision, Recall, F1, Accuracy, ROC-AUC, PR-AUC and the confusion matrix are reported.
- Feature importance and a Decision Tree visualization support interpretability.
- The complete preprocessing + model pipeline is saved for API reuse.
